# First Data exploration

In [2]:
#import packages
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'pandas'

### 1. Load electricity data

In [ ]:

ROOT = Path.cwd()

if ROOT.name == "code":
    ROOT = ROOT.parent

DATA = ROOT / "data" / "processed"
FIGURES = ROOT / "output" / "figures"

FIGURES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    DATA / "electricity_daily.csv",
    parse_dates=["date"]
)

df = df.sort_values("date").reset_index(drop=True)

print("First date:", df["date"].min())
print("Last date:", df["date"].max())
print("Total observations:", len(df))

display(df.head())

### 2. Plot the complete electricity consumption time series

df["rolling_30d"] = (
    df["consumption_mwh"]
    .rolling(window=30)
    .mean()
)

df["rolling_365d"] = (
    df["consumption_mwh"]
    .rolling(window=365)
    .mean()
)

plt.figure(figsize=(15, 6))

plt.plot(
    df["date"],
    df["consumption_mwh"],
    label="Daily consumption",
    alpha=0.25
)

plt.plot(
    df["date"],
    df["rolling_30d"],
    label="30-day moving average",
    linewidth=1.5
)

plt.plot(
    df["date"],
    df["rolling_365d"],
    label="365-day moving average",
    linewidth=2
)

plt.title(
    "Historical Electricity Consumption in Basel-Stadt"
)
plt.xlabel("Date")
plt.ylabel("Electricity consumption (MWh/day)")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    FIGURES / "01_electricity_full_series.png",
    dpi=300
)

plt.show()

### 3. Compare annual electricity consumption

#### 3.1 annual consumption

In [ ]:
df["year"] = df["date"].dt.year

annual = (
    df.groupby("year")
    .agg(
        average_consumption=("consumption_mwh", "mean"),
        total_consumption=("consumption_mwh", "sum"),
        days=("consumption_mwh", "count")
    )
)

annual["change_pct"] = (
    annual["average_consumption"].pct_change() * 100
)

display(annual.round(2))

#### 3.2 Annual average consumption chart

In [ ]:
# Only compare complete calendar years
complete_years = annual[
    annual.index.map(
        lambda year: (
            pd.Timestamp(year + 1, 1, 1)
            - pd.Timestamp(year, 1, 1)
        ).days
    ) == annual["days"]
]

plt.figure(figsize=(12, 5))

plt.bar(
    complete_years.index.astype(str),
    complete_years["average_consumption"]
)

plt.title(
    "Average Daily Electricity Consumption by Year"
)
plt.xlabel("Year")
plt.ylabel("Average consumption (MWh/day)")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()

plt.savefig(
    FIGURES / "02_annual_average_consumption.png",
    dpi=300
)

plt.show()

## 4. Examine monthly seasonality

In [ ]:
df["month"] = df["date"].dt.month

monthly = (
    df.groupby("month")["consumption_mwh"]
    .mean()
)

plt.figure(figsize=(10, 5))

plt.bar(monthly.index, monthly.values)

plt.title(
    "Average Daily Electricity Consumption by Month"
)
plt.xlabel("Month")
plt.ylabel("Average consumption (MWh/day)")
plt.xticks(range(1, 13))
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()

plt.savefig(
    FIGURES / "03_monthly_seasonality.png",
    dpi=300
)

plt.show()